# Flexible Flow Shop Scheduling Problem (FFSP) - 柔性流水车间

## 问题定义
柔性流水车间 (FFS) 是 Flow Shop 与 Flexible Job Shop 的结合：
- 所有工件经过 **相同的 Stage 序列**（像 Flow Shop）
- 每道 Stage 有 **多台并行机**，可选择的机器不同加工时间不同（像 Flexible Job Shop）

### 与 FJSP 的区别
- **FJSP**: 每个工件有自己的加工路线（工艺路线不同）
- **FFS**: 所有工件共享同一 Stage 顺序（Stage1 → Stage2 → ... → Stagek）

### 动态调度场景
实际生产中会面临：
1. 新工件随机到达
2. 机器突发故障/维修
3. 订单取消/插入急单
4. 加工时间波动

本实现采用 **事件驱动 + 周期重调度** 混合策略。


In [ ]:
import sys
sys.path.insert(0, r'E:\7.10\code')
from FFS_DynamicScheduler import *
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import copy

## 生成 FFS 问题实例
- `machines_per_stage=[2, 3, 2]` 表示 3 个 Stage，分别有 2、3、2 台并行机
- 每个工件在每阶段的每台机器上有不同的加工时间 (3-15)

In [ ]:
# 生成 5 个工件、3 个阶段的静态 FFS 问题
stage_machines, P, total_machines = generate_ffs_problem(
    n_jobs=5, n_stages=3, machines_per_stage=[2, 3, 2], seed=100)

print_problem_info(stage_machines, P, 'FFS 静态问题')
print(f'\n加工时间矩阵 P[job][stage][machine]:')
for j in range(len(P)):
    print(f'  J{j}: ', end='')
    for s in range(len(P[j])):
        print(f'  S{s}{P[j][s]}', end='')
    print()

## 静态 GA 求解 FFS

编码策略与 FJSP 一致：**MS (Machine Selection) + OS (Operation Sequence)**

关键差异：
- MS 长度 = n_jobs × n_stages（每道工序选同一 stage 内的一台机器）
- 解码器强制工件按 Stage 0 → Stage 1 → ... 顺序加工
- 交叉/变异与 FJSP 相同（MS 两点交叉 + OS POX 交叉）

In [ ]:
# GA 求解静态 FFS
sched_ga, mk_ga, chrom, history = ga_ffs(
    stage_machines, P,
    pop_size=80, generations=120, pc=0.8, pm=0.15)

print(f'\nGA 调度最优 Makespan = {mk_ga}')
draw_convergence(history, 'FFS-GA 收敛曲线')

In [ ]:
# 绘制最优调度甘特图
draw_ffs_gantt(sched_ga, stage_machines, len(P),
               f'FFS-GA 最优调度 (Makespan={mk_ga})')

## 派工规则对比
作为对比基准，实现了几种经典派工规则（不需要 GA 迭代，计算极快）：

In [ ]:
# 各派工规则对比
print('各派工规则 Makespan 对比:')
print('-' * 40)
results = []
for rule in ['SPT', 'MWKR', 'LWKR', 'FIFO']:
    sched, mk = dispatch_schedule(stage_machines, P, rule)
    results.append((rule, mk))
    print(f'  {rule:5s}: Makespan = {mk}')

# 可视化对比
rules, mks = zip(*results)
plt.figure(figsize=(10, 5))
bars = plt.bar(rules, mks, color=['#4CAF50','#2196F3','#FF9800','#9C27B0'])
plt.ylabel('Makespan', fontsize=12)
plt.title('派工规则 Makespan 对比', fontsize=14)
for bar, mk in zip(bars, mks):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(mk), ha='center', va='bottom', fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 动态调度模拟

生成包含动态事件的 FFS 问题：
- 3 个初始工件 (J0, J1, J2)
- 2 个在 t=34, t=46 到达的动态工件 (J3, J4)
- 机器 M2 在 t=50 发生故障，持续 12 时间单位

调度引擎 (`DynamicFFSEngine`) 采用**事件驱动 + 周期重调度**策略。

In [ ]:
# 生成动态 FFS 问题 (含初始工件 + 动态到达 + 故障)
stage_machines, P_initial, total_machines, dynamic_jobs, breakdowns = \
    generate_dynamic_ffs(n_initial_jobs=3, n_stages=2,
                         machines_per_stage=[2, 2],
                         n_dynamic_jobs=2, seed=42)

print_problem_info(stage_machines, P_initial, '动态 FFS 问题')

print(f'\n动态到达的工件:')
for dj in dynamic_jobs:
    print(f'  J{dj["id"]}: 到达 t={dj["arrival"]}, '
          f'加工时间 {[f"S{s}{t}" for s,t in enumerate(dj["processing"])]}')

print(f'\n机器故障:')
for bd in breakdowns:
    print(f'  M{bd["machine"]}: t={bd["start"]}~{bd["end"]} ',
          f'(持续{bd["duration"]})')

In [ ]:
# GA 动态调度
engine_ga = DynamicFFSEngine(stage_machines, P_initial,
                              schedule_rule='GA', reschedule_interval=30)
for dj in dynamic_jobs:
    engine_ga.add_job_arrival(dj['arrival'], dj['processing'])
for bd in breakdowns:
    engine_ga.add_breakdown(bd['machine'], bd['start'], bd['duration'])

final_sched_ga = engine_ga.run_simulation(max_time=100)

if final_sched_ga:
    n_all = len(P_initial) + len(dynamic_jobs)
    final_mk = max(e for _,_,_,_,e in final_sched_ga)
    draw_ffs_gantt(final_sched_ga, stage_machines, n_all,
                   f'GA 动态调度 (最终 Makespan={final_mk})')

draw_makespan_timeline(engine_ga.makespan_history,
                       'GA 动态调度 Makespan 变化')

In [ ]:
# SPT 动态调度（对比）
engine_spt = DynamicFFSEngine(stage_machines, P_initial,
                               schedule_rule='SPT', reschedule_interval=30)
for dj in dynamic_jobs:
    engine_spt.add_job_arrival(dj['arrival'], dj['processing'])
for bd in breakdowns:
    engine_spt.add_breakdown(bd['machine'], bd['start'], bd['duration'])

final_sched_spt = engine_spt.run_simulation(max_time=100)

if final_sched_spt:
    n_all = len(P_initial) + len(dynamic_jobs)
    final_mk = max(e for _,_,_,_,e in final_sched_spt)
    draw_ffs_gantt(final_sched_spt, stage_machines, n_all,
                   f'SPT 动态调度 (最终 Makespan={final_mk})')

draw_makespan_timeline(engine_spt.makespan_history,
                       'SPT 动态调度 Makespan 变化')

## 总结与扩展

### 当前实现的核心能力：
1. **静态 FFS-GA 调度** — 与 FJSP 同样的 MS+OS 编码框架，适配 FFS 的 Stage 约束
2. **派工规则调度** — SPT/MWKR/LWKR/FIFO 多种规则快速求解
3. **动态事件处理** — 新工件到达 + 机器故障的事件驱动重调度
4. **周期重调度** — 可配置间隔的定期优化

### 可扩展方向：
- **多目标优化**：Makespan + 总延迟 + 能耗多目标 NSGA-II
- **有限缓冲区**：Stage 间在制品容量约束
- **Sequence-dependent Setup Time**：同机器上前后工序的切换时间
- **Transit Time**：工件在不同 Stage 间的运输时间
- **机器学习**：用 GA 进化的 GP 树替代固定派工规则（参考 FJSP-GP.ipynb 的思路）